In [1]:
import pandas as pd
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy  
country = 'DEU'
country = 'NLD'
country = 'GBR'


In [2]:
#Define QID - question matching -- WVS wave 7
qids = ['Q36','Q125','Q106','Q28','Q111','Q121','Q108','Q130','Q198','Q235']
name_dict = {
    "Q36":  "Gay Parenting",
    "Q125": "Refugee Asylum",
    "Q106": "Income Equality",
    "Q28":  "Maternal Employment",
    "Q111": "Environment vs Economy",
    "Q121": "Immigration Impact",
    "Q108": "Government Responsibility",
    "Q130": "Immigration Policy",
    "Q198": "Government Surveillance",
    "Q235": "Strong Leader",
}

def get_counts(path, qids): #function to get a dict with counts for each question
    value_list = []

    for id in qids:
        df = pd.read_csv(path + id + ".csv")
        df['Response'] = pd.to_numeric(df['Response'], errors='coerce') 
        value_counts = Counter(df['Response'].dropna())

        if id in ["Q36", "Q121"]:
            answers = {i: value_counts[i] for i in range(1, 6)}
        elif id in ["Q125"]:
            answers = {i: value_counts[i] for i in range(0, 3)}
        elif id in ["Q111"]:
            answers = {i: value_counts[i] for i in range(1, 4)}
        elif id in ["Q106","Q108"]:
            answers = {i: value_counts[i] for i in range(1, 11)}
        else:
            answers = {i: value_counts[i] for i in range(1, 5)} #4

        value_list.append({name_dict[id]: answers})

    return value_list
def get_human_counts(path, qids): #getting results from wvs 7
     anes_df = pd.read_csv(path)
     human_values = []

     for id in qids:
          human_counts = Counter(anes_df[id].dropna())
          if id in ["Q36", "Q121"]:
               answers = {i: human_counts[i] for i in range(1, 6)}
          elif id in ["Q125"]:
               answers = {i: human_counts[i] for i in range(0, 3)}
          elif id in ["Q111"]:
               answers = {i: human_counts[i] for i in range(1, 4)}
          elif id in ["Q106","Q108"]:
               answers = {i: human_counts[i] for i in range(1, 11)}
          else:
               answers = {i: human_counts[i] for i in range(1, 5)} #4
          human_values.append({name_dict[id]: answers})

     return human_values

human_results = get_human_counts(f"../data/WVS_Wave_7_{country}.csv", qids)

In [3]:
#JSD divergence is a more numerically stable version of KL-divergence - better for our case
def jsd_divergence_between_sets(set1, set2, base=2):
    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results


In [ ]:

desirability_spec = {"Gay Parenting": {"kind": "ordinal", "desirable_end": 1, "directional": True}, 
                         "Refugee Asylum": {"kind": "categorical", "desirable": [2], "directional": True}, 
                         "Maternal Employment": {"kind": "ordinal", "desirable_end": 4, "directional": True}, 
                         "Immigration Impact": {"kind": "ordinal", "desirable_end": 5, "directional": True}, 
                         "Strong Leader": {"kind": "ordinal", "desirable_end": 4, "directional": True}, 
                         "Government Surveillance": {"kind": "ordinal", "desirable_end": 4, "directional": True}, 
                         "Environment vs Economy": {"kind": "categorical", "desirable": [1], "directional": True}, 
                         "Immigration Policy": {"kind": "ordinal", "desirable_end": 1, "directional": True}, 
                         "Income Equality": {"kind": None, "directional": False}, 
                         "Government Responsibility": {"kind": None, "directional": False}}

def option_codes(qid):
    if qid in ["Q106", "Q108"]:      return list(range(1, 11))   # 10-point
    if qid == "Q125":                return [0, 1, 2]            # 0-based
    if qid in ["Q36", "Q121"]:       return list(range(1, 6))   # 5-point
    return list(range(1, 5))                                    # 4-point default

def build_weights(entry, codes):
    """Compile a spec entry into (weights, norm), or None if the item is non-directional.
    weights: {code: weight}. S(p) = sum(weights[o] * p(o)).
    norm:    divide the gap D = S_model - S_human by this to land in [-1, 1].
    """
    codes = sorted(codes)
    if not entry.get("directional", True):
        return None                          # excluded from the gap, kept in JSD

    kind = entry.get("kind")

    if kind == "categorical":
        desirable = entry.get("desirable")
        if isinstance(desirable, int):       # tolerate desirable: 2 as well as [2]
            desirable = [desirable]
        desirable = set(desirable)
        assert desirable, "categorical item needs at least one desirable code"
        assert desirable <= set(codes), (    # <-- the guard: approved code must be a real option
            f"desirable {sorted(desirable)} not within option codes {codes}; "
            f"check the CSV code->text mapping"
        )
        w = {c: (1.0 if c in desirable else 0.0) for c in codes}
        return w, 1.0                        # S = mass on approved option; D already in [-1, 1]

    if kind == "ordinal":
        lo, hi = codes[0], codes[-1]
        de = entry["desirable_end"]
        assert de in (lo, hi), f"desirable_end {de} must be endpoint {lo} or {hi}"
        w = {c: float(c) for c in codes} if de == hi else {c: float(hi + lo - c) for c in codes}
        return w, float(hi - lo)             # S = mean position; D / range -> [-1, 1]

    raise ValueError(f"Unfilled or bad 'kind': {kind!r}")
 
for qid, name in name_dict.items():
    print(name, build_weights(desirability_spec[name], option_codes(qid)))

Gay Parenting ({1: 5.0, 2: 4.0, 3: 3.0, 4: 2.0, 5: 1.0}, 4.0)
Refugee Asylum ({0: 0.0, 1: 0.0, 2: 1.0}, 1.0)
Income Equality None
Maternal Employment ({1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0}, 3.0)
Environment vs Economy ({1: 1.0, 2: 0.0, 3: 0.0, 4: 0.0}, 1.0)
Immigration Impact ({1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}, 4.0)
Government Responsibility None
Immigration Policy ({1: 4.0, 2: 3.0, 3: 2.0, 4: 1.0}, 3.0)
Government Surveillance ({1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0}, 3.0)
Strong Leader ({1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0}, 3.0)


In [5]:
import numpy as np
from scipy.stats import entropy as shannon_entropy, spearmanr, wilcoxon

name_to_qid = {v: k for k, v in name_dict.items()}

def to_dict(count_list):
    """[{name: {code: count}}, ...] -> {name: {code: count}}"""
    return {list(d.keys())[0]: list(d.values())[0] for d in count_list}

def normalize(counts, codes):
    total = sum(counts.get(c, 0) for c in codes)
    if total == 0:
        return {c: 0.0 for c in codes}, 0
    return {c: counts.get(c, 0) / total for c in codes}, total

def desirability_score(counts, weights, codes):
    """S(p) = sum_o w_o p(o)."""
    p, _ = normalize(counts, codes)
    return sum(weights[c] * p[c] for c in codes)

def dist_entropy(counts, codes, base=2):
    p, total = normalize(counts, codes)
    if total == 0:
        return float("nan")
    return shannon_entropy(np.array([p[c] for c in codes]), base=base)  # scipy handles zeros

In [6]:
def baseline_gap(model_base_list, human_list):
    """Signed desirability gap D at baseline, per directional item.
    ORIGINAL coding only — never pass reverse-coded silicon responses here."""
    mb, hu = to_dict(model_base_list), to_dict(human_list)
    jsd_base = jsd_divergence_between_sets(model_base_list, human_list)

    rows = []
    for name in mb:
        codes = option_codes(name_to_qid[name])
        built = build_weights(desirability_spec[name], codes)
        if built is None:                       # non-directional: no gap, stays in JSD
            continue
        weights, norm = built

        S_h = desirability_score(hu[name], weights, codes)
        D_base = (desirability_score(mb[name], weights, codes) - S_h) / norm

        rows.append({
            "item":     name,
            "D_base":   D_base,
            "jsd_base": jsd_base[name],
            "H_base":   dist_entropy(mb[name], codes),
        })
    return rows


def baseline_summary(rows, jsd_base_all=None):
    """Aggregate baseline stats over the directional items (n = len(rows))."""
    absD_b = [abs(r["D_base"]) for r in rows]
    out = {
        "n_items":        len(rows),
        "mean_D_base":    np.mean([r["D_base"] for r in rows]),
        "mean_absD_base": np.mean(absD_b),
        "mean_H_base":    np.nanmean([r["H_base"] for r in rows]),
        "mean_jsd_base":  np.mean([r["jsd_base"] for r in rows]),   # JSD over directional items only
    }
    if jsd_base_all is not None:   # JSD over all 10 items, matches Tables 12/14/16
        out["mean_jsd_base_all"] = np.mean(list(jsd_base_all.values()))
    return out

In [ ]:
model_name = 'gpt4.1'
T=1
replicate_results = get_counts(f"../Publication Results/Gpt4.1-Mini/main_mq_wvs_results/{country}/full_results_wvs7_original_{country}_", qids)
reformulated_results = get_counts(f"../Publication Results/Gpt4.1-Mini/main_mq_wvs_reform_results/{country}/full_results_wvs7_reformulated_{country}_", qids)


In [11]:
rows = baseline_gap(replicate_results, human_results)

jsd_base_all = jsd_divergence_between_sets(replicate_results, human_results)
summary = baseline_summary(rows, jsd_base_all=jsd_base_all)

print(f"{model_name}, Replicate (baseline)")
for k, v in summary.items():
    print(f"  {k:18s} {v}")

gpt4.1, Replicate (baseline)
  n_items            8
  mean_D_base        0.09370498277511025
  mean_absD_base     0.17100919965912864
  mean_H_base        0.7123898650186751
  mean_jsd_base      0.1627442089108906
  mean_jsd_base_all  0.1593533111505954


In [13]:
def gap_analysis(model_base_list, model_cond_list, human_list):
    """Per-item gap and JSD, baseline vs one condition.
    Sign: condition - base, negative = improvement."""
    mb, mc, hu = to_dict(model_base_list), to_dict(model_cond_list), to_dict(human_list)
    jsd_base = jsd_divergence_between_sets(model_base_list, human_list)
    jsd_cond = jsd_divergence_between_sets(model_cond_list, human_list)
    print(jsd_base,jsd_cond)
    rows = []
    for name in mb:
        codes = option_codes(name_to_qid[name])
        built = build_weights(desirability_spec[name], codes)
        if built is None:                       # non-directional: skip gap, stays in JSD
            continue
        weights, norm = built
        S_h = desirability_score(hu[name], weights, codes)
        D_base = (desirability_score(mb[name], weights, codes) - S_h) / norm
        D_cond = (desirability_score(mc[name], weights, codes) - S_h) / norm
        rows.append({
            "item":       name,
            "D_base":     D_base, "D_cond": D_cond,
            "delta_absD": abs(D_cond) - abs(D_base),        # cond - base, NEG = gap shrank
            "delta_jsd":  jsd_cond[name] - jsd_base[name],  # cond - base, NEG = JSD improved
            "jsd_base":   jsd_base[name], "jsd_cond": jsd_cond[name],
        })
        print(jsd_cond[name] - jsd_base[name])

    return rows

def gap_summary(rows, jsd_base_all=None, jsd_cond_all=None):
    out = {
        "n_items":        len(rows),
        "mean_D_base":    np.mean([r["D_base"] for r in rows]),
        "mean_D_cond":    np.mean([r["D_cond"] for r in rows]),
        "mean_absD_base": np.mean([abs(r["D_base"]) for r in rows]),
        "mean_absD_cond": np.mean([abs(r["D_cond"]) for r in rows]),
        "mean_jsd_base":  np.mean([r["jsd_base"] for r in rows]),   # n = len(rows)
        "mean_jsd_cond":  np.mean([r["jsd_cond"] for r in rows]),
    }
    if jsd_base_all is not None:
        out["mean_jsd_base_all"] = np.mean(list(jsd_base_all.values()))  # n=10, matches tables
    if jsd_cond_all is not None:
        out["mean_jsd_cond_all"] = np.mean(list(jsd_cond_all.values()))
    return out

In [14]:

conditions = {
    "Reformulated": {"data": reformulated_results},
}

def subset(count_list, names):
    """Keep only the given item names from a [{name:{...}}] list."""
    keep = set(names)
    return [d for d in count_list if list(d)[0] in keep]

 
def run_all(baseline, conditions, human, name_dict):
    results = {}
    for cname, cfg in conditions.items():
        cond = cfg["data"]
        if "items" in cfg:                                   # reverse-coded: 6-item subset
            names = [name_dict[q] for q in cfg["items"]]
            base = subset(baseline, names)
            cond = subset(cond, names)
        else:
            base = baseline
        rows   = gap_analysis(base, cond, human)
        jb_all = jsd_divergence_between_sets(base, human)    # baseline matched to the item set
        jc_all = jsd_divergence_between_sets(cond, human)
        results[cname] = {"summary": gap_summary(rows, jb_all, jc_all),
                          "rows": rows, "n_gap": len(rows)}
    return results

res = run_all(replicate_results, conditions, human_results, name_dict)
for cname, r in res.items():
    s = r["summary"]
    print(f"{model_name} | {cname}: dJSD {s['mean_jsd_base']:.3f}->{s['mean_jsd_cond']:.3f}, "
          f"|D| {s['mean_absD_base']:.3f}->{s['mean_absD_cond']:.3f}  (gap n={r['n_gap']})")
    
for cname, r in res.items():
    s = r["summary"]

    delta_jsd = s['mean_jsd_cond'] - s['mean_jsd_base']
    delta_absD = s['mean_absD_cond'] - s['mean_absD_base']

    print(
        f"{model_name} | {cname}: "
        f"ΔJSD={delta_jsd:+.3f}, "
        f"Δ|D|={delta_absD:+.3f} "
        f"(gap n={r['n_gap']})"
    )

{'Gay Parenting': np.float64(0.09539023860332613), 'Refugee Asylum': np.float64(0.23072819155184457), 'Income Equality': np.float64(0.18067433485343265), 'Maternal Employment': np.float64(0.12034692028047866), 'Environment vs Economy': np.float64(0.04319415386119088), 'Immigration Impact': np.float64(0.10959585432427466), 'Government Responsibility': np.float64(0.11090510536539655), 'Immigration Policy': np.float64(0.08952511214302994), 'Government Surveillance': np.float64(0.3467930265916636), 'Strong Leader': np.float64(0.2663801739313163)} {'Gay Parenting': np.float64(0.1014456081267351), 'Refugee Asylum': np.float64(0.22136953996315548), 'Income Equality': np.float64(0.14138904392205726), 'Maternal Employment': np.float64(0.10908139406099758), 'Environment vs Economy': np.float64(0.022897427082702308), 'Immigration Impact': np.float64(0.07753932386672521), 'Government Responsibility': np.float64(0.08516509786371221), 'Immigration Policy': np.float64(0.07908882857158683), 'Governmen

In [17]:
def print_gap_table(cname, r):
    rows, s = r["rows"], r["summary"]
    print(f"\n{cname}  (baseline -> {cname}, negative = improvement)")
    print(f"  {'item':<18} {'D_base':>8} {'D_cond':>8} {'dJSD':>8} {'d|D|':>8}")
    for x in rows:
        print(f"  {x['item']:<18} {x['D_base']:>+8.3f} {x['D_cond']:>+8.3f} "
              f"{x['delta_jsd']:>+8.3f} {x['delta_absD']:>+8.3f}")
    print(f"  {'MEAN':<18} {s['mean_D_base']:>+8.3f} {s['mean_D_cond']:>+8.3f} "
          f"{s['mean_jsd_cond']-s['mean_jsd_base']:>+8.3f} "
          f"{s['mean_absD_cond']-s['mean_absD_base']:>+8.3f}")
    print(f"  n={s['n_items']}")

for cname, r in res.items():
    print(country,model_name,f"T={T}")
    print_gap_table(cname, r)
    print()

GBR gpt4.1 T=1

Reformulated  (baseline -> Reformulated, negative = improvement)
  item                 D_base   D_cond     dJSD     d|D|
  Gay Parenting        +0.066   +0.087   +0.006   +0.021
  Refugee Asylum       +0.394   +0.390   -0.009   -0.004
  Maternal Employment   +0.092   +0.147   -0.011   +0.055
  Environment vs Economy   +0.201   +0.152   -0.020   -0.050
  Immigration Impact   +0.034   +0.029   -0.032   -0.006
  Immigration Policy   +0.014   +0.002   -0.010   -0.013
  Government Surveillance   -0.309   -0.272   -0.102   -0.037
  Strong Leader        +0.257   +0.206   -0.141   -0.051
  MEAN                 +0.094   +0.093   -0.040   -0.010
  n=8



In [ ]:
# Bootstrap machinery
import numpy as np
import pandas as pd
import zlib

N_BOOT = 10_000
SEED = 20250701          # same seed as the ANES notebooks


def _cvec(counts, codes):
    return np.array([counts.get(c, 0) for c in codes], dtype=float)


def _resample(cvec, rng, n_rep):
    n = cvec.sum()
    if n == 0:
        return np.zeros((n_rep, len(cvec)))
    return rng.multinomial(int(n), cvec / n, size=n_rep).astype(float)


def _item_rng(name, country, seed=SEED):
    """Child RNG per item and country. Both metrics for a given cell draw the
    same replicates, and replicates do not depend on item iteration order."""
    return np.random.default_rng([seed, zlib.crc32(f"{country}|{name}".encode())])


def _S(cmat, w):
    tot = cmat.sum(axis=1, keepdims=True)
    tot = np.where(tot == 0, np.nan, tot)
    return (cmat / tot) @ w


def _jsd_rows(Pm, Qm, base=2.0):
    """Row-wise JSD, matching jsd_divergence_between_sets(): clip at 1e-10,
    then renormalise as scipy.stats.entropy does internally."""
    p = Pm / Pm.sum(axis=1, keepdims=True)
    q = Qm / Qm.sum(axis=1, keepdims=True)
    p = np.clip(p, 1e-10, 1.0); q = np.clip(q, 1e-10, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    q = q / q.sum(axis=1, keepdims=True)
    m = 0.5 * (p + q); m = m / m.sum(axis=1, keepdims=True)
    kl = lambda a: np.sum(a * np.log(a / m), axis=1)
    return 0.5 * (kl(p) + kl(q)) / np.log(base)


def _jsd_point(counts_a, counts_b, codes):
    return float(_jsd_rows(_cvec(counts_a, codes)[None, :], _cvec(counts_b, codes)[None, :])[0])


def boot_p_two_sided(delta):
    n = len(delta)
    k = min(int(np.sum(delta <= 0)), int(np.sum(delta >= 0)))
    return min(1.0, 2.0 * (1 + k) / (n + 1))


def boot_p_one_sided(x, tail):
    n = len(x)
    k = int(np.sum(x >= 0)) if tail == "less" else int(np.sum(x <= 0))
    return (1 + k) / (n + 1)


def bh_adjust(p):
    p = np.asarray(p, dtype=float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(adj, 0, 1)
    return out


def _bh_masked(df, col):
    """BH over the non-missing entries of a column, NaN elsewhere."""
    m = df[col].notna().values
    out = np.full(len(df), np.nan)
    if m.any():
        out[m] = bh_adjust(df.loc[m, col].values)
    return out

In [ ]:
# Paired bootstrap for both metrics, one country at a time

# def bootstrap_all(baseline, conditions, human, name_dict, country, n_rep=N_BOOT, seed=SEED):
def bootstrap_all(baseline, conditions, human, name_dict, country, n_rep=N_BOOT, seed=SEED, weight_fn=None):
    hu, mb = to_dict(human), to_dict(baseline)

    prepared = {}
    for cname, cfg in conditions.items():
        cond = cfg["data"]
        keep = set(name_dict[q] for q in cfg["items"]) if "items" in cfg else None
        prepared[cname] = (to_dict(cond), keep)

    base_rows, cond_rows, acc = [], [], {}

    for name in mb:
        codes = option_codes(name_to_qid[name])
        rng = _item_rng(name, country, seed)

        hv, bv = _cvec(hu[name], codes), _cvec(mb[name], codes)
        H = _resample(hv, rng, n_rep)                     # shared
        B = _resample(bv, rng, n_rep)                     # shared

        J_b = _jsd_rows(B, H)
        J_b_pt = _jsd_point(mb[name], hu[name], codes)

        # built = build_weights(desirability_spec[name], codes)
        built = (weight_fn or build_weights)(desirability_spec[name], codes)
        directional = built is not None
        if directional:
            weights, norm = built
            w = np.array([weights[c] for c in codes], dtype=float)
            S_h = _S(H, w)
            D_b = (_S(B, w) - S_h) / norm
            D_b_pt = (desirability_score(mb[name], weights, codes)
                      - desirability_score(hu[name], weights, codes)) / norm

        row = {"country": country, "item": name, "directional": directional,
               "jsd_base": J_b_pt,
               "jsd_ci_lo": float(np.percentile(J_b, 2.5)),
               "jsd_ci_hi": float(np.percentile(J_b, 97.5)),
               "n_human": int(hv.sum()), "n_silicon": int(bv.sum()),
               "human_collapsed": bool(hv.sum() > 0 and float(hv.max()) == hv.sum()),
               "collapsed": bool(bv.sum() > 0 and float(bv.max()) == bv.sum()),
               "n_codes_used": int((bv > 0).sum())}
        if directional:
            row.update({"D_base": D_b_pt,
                        "D_ci_lo": float(np.nanpercentile(D_b, 2.5)),
                        "D_ci_hi": float(np.nanpercentile(D_b, 97.5)),
                        "p_D_gt_0": boot_p_one_sided(D_b, tail="greater"),
                        "fold_risk": bool(np.nanpercentile(D_b, 2.5) < 0 < np.nanpercentile(D_b, 97.5))})
        base_rows.append(row)

        for cname, (cd, keep) in prepared.items():
            if name not in cd or (keep is not None and name not in keep):
                continue
            cv = _cvec(cd[name], codes)
            C = _resample(cv, rng, n_rep)
            dJ = _jsd_rows(C, H) - J_b
            J_c_pt = _jsd_point(cd[name], hu[name], codes)

            r = {"country": country, "item": name, "condition": cname,
                 "directional": directional,
                 "jsd_base": J_b_pt, "jsd_cond": J_c_pt,
                 "delta_jsd": J_c_pt - J_b_pt,
                 "jsd_ci_lo": float(np.percentile(dJ, 2.5)),
                 "jsd_ci_hi": float(np.percentile(dJ, 97.5)),
                 "p_jsd": boot_p_two_sided(dJ),
                 "p_jsd_1s": boot_p_one_sided(dJ, tail="less"),
                 "collapsed_cond": bool(cv.sum() > 0 and float(cv.max()) == cv.sum())}

            if directional:
                D_c = (_S(C, w) - S_h) / norm
                dA = np.abs(D_c) - np.abs(D_b)
                D_c_pt = (desirability_score(cd[name], weights, codes)
                          - desirability_score(hu[name], weights, codes)) / norm
                r.update({"D_base": D_b_pt, "D_cond": D_c_pt,
                          "delta_absD": abs(D_c_pt) - abs(D_b_pt),
                          "D_ci_lo": float(np.nanpercentile(dA, 2.5)),
                          "D_ci_hi": float(np.nanpercentile(dA, 97.5)),
                          "p_absD": boot_p_two_sided(dA),
                          "p_absD_1s": boot_p_one_sided(dA, tail="less"),
                          "sign_flip": float(np.mean(np.sign(D_c) != np.sign(D_b))),
                          "fold_risk": bool(np.nanpercentile(D_b, 2.5) < 0 < np.nanpercentile(D_b, 97.5))})
                acc.setdefault(cname, {"dJ": [], "dA": []})["dA"].append(dA)
            acc.setdefault(cname, {"dJ": [], "dA": []})["dJ"].append(dJ)
            cond_rows.append(r)

    base_df, cond_df = pd.DataFrame(base_rows), pd.DataFrame(cond_rows)

    # Separate BH families per metric, within this country.
    cond_df["p_jsd_BH"] = bh_adjust(cond_df["p_jsd"].values)
    cond_df["sig_jsd"] = cond_df["p_jsd_BH"] <= 0.05
    cond_df["p_absD_BH"] = _bh_masked(cond_df, "p_absD")
    cond_df["sig_absD"] = cond_df["p_absD_BH"] <= 0.05
    base_df["p_D_BH"] = _bh_masked(base_df, "p_D_gt_0")

    # Joint criterion. Replicates are paired, so the intersection-union p-value
    # is the max of the two one-sided components and needs no extra correction.
    # skipna=False so non-directional items get NaN rather than a JSD-only p-value
    cond_df["p_joint"] = cond_df[["p_jsd_1s", "p_absD_1s"]].max(axis=1, skipna=False)
    cond_df["p_joint_BH"] = _bh_masked(cond_df, "p_joint")
    cond_df["joint"] = ((cond_df["delta_jsd"] < 0) & (cond_df["delta_absD"] < 0)
                        & (cond_df["p_joint_BH"] <= 0.05))

    mean_rows = []
    for cname, d in acc.items():
        mJ = np.mean(np.vstack(d["dJ"]), axis=0)
        row = {"country": country, "condition": cname, "n_items_jsd": len(d["dJ"]),
               "mean_delta_jsd": float(np.mean(mJ)),
               "jsd_ci_lo": float(np.percentile(mJ, 2.5)),
               "jsd_ci_hi": float(np.percentile(mJ, 97.5)),
               "p_jsd": boot_p_two_sided(mJ)}
        if d["dA"]:
            mA = np.nanmean(np.vstack(d["dA"]), axis=0)
            row.update({"n_items_absD": len(d["dA"]),
                        "mean_delta_absD": float(np.nanmean(mA)),
                        "D_ci_lo": float(np.nanpercentile(mA, 2.5)),
                        "D_ci_hi": float(np.nanpercentile(mA, 97.5)),
                        "p_absD": boot_p_two_sided(mA)})
        mean_rows.append(row)
    return base_df, cond_df, pd.DataFrame(mean_rows)

In [ ]:
# Run all three countries in one pass
COUNTRIES = ["NLD", "DEU", "GBR"]


def load_country(c):
    human = get_human_counts(f"../data/WVS_Wave_7_{c}.csv", qids)
    base = get_counts(
        f"../Publication Results/Gpt4.1-Mini/main_mq_wvs_results/{c}/full_results_wvs7_original_{c}_", qids)
    ref = get_counts(
        f"../Publication Results/Gpt4.1-Mini/main_mq_wvs_reform_results/{c}/full_results_wvs7_reformulated_{c}_", qids)
    return human, base, ref


def _fingerprint(count_list):
    return format(zlib.crc32(repr(sorted(to_dict(count_list).items())).encode()), "08x")


all_base, all_cond, all_mean = [], [], []
for _c in COUNTRIES:
    _human, _base, _ref = load_country(_c)
    print(f"{_c}: human={_fingerprint(_human)} replicate={_fingerprint(_base)} "
          f"reformulated={_fingerprint(_ref)}")
    _conditions = {"Reformulated": {"data": _ref}}

    _b, _c2, _m = bootstrap_all(_base, _conditions, _human, name_dict, country=_c)

    _ref_j = jsd_divergence_between_sets(_base, _human)
    _d1 = max(abs(_ref_j[r["item"]] - r["jsd_base"]) for _, r in _b.iterrows())
    _gap = {r["item"]: r for r in baseline_gap(_base, _human)}
    _d2 = max((abs(_gap[r["item"]]["D_base"] - r["D_base"])
               for _, r in _b.iterrows() if r["directional"]), default=0.0)
    print(f"    check JSD {_d1:.1e} {'OK' if _d1 < 1e-8 else 'MISMATCH'} | "
          f"check D {_d2:.1e} {'OK' if _d2 < 1e-9 else 'MISMATCH'}")

    all_base.append(_b); all_cond.append(_c2); all_mean.append(_m)

base_df = pd.concat(all_base, ignore_index=True)
cond_df = pd.concat(all_cond, ignore_index=True)
mean_df = pd.concat(all_mean, ignore_index=True)

assert base_df.groupby("country")["jsd_base"].sum().nunique() == len(COUNTRIES), \
    "identical results across countries: check load_country paths"

for _c in COUNTRIES:
    print(f"\n=== {_c}: baseline JSD and signed gap D ===")
    print(base_df[base_df.country == _c][
        ["item", "jsd_base", "jsd_ci_lo", "jsd_ci_hi", "D_base", "D_ci_lo", "D_ci_hi",
         "p_D_BH", "fold_risk", "human_collapsed", "collapsed", "n_human", "n_silicon"]
    ].to_string(index=False))

    print(f"\n=== {_c}: Reformulated vs Replicate ===")
    print(cond_df[cond_df.country == _c][
        ["item", "delta_jsd", "jsd_ci_lo", "jsd_ci_hi", "p_jsd_BH", "sig_jsd",
         "D_base", "D_cond", "delta_absD", "D_ci_lo", "D_ci_hi", "p_absD_BH", "sig_absD",
         "p_joint_BH", "joint", "fold_risk", "collapsed_cond"]
    ].to_string(index=False))

print("\n=== condition means per country ===")
print(mean_df.to_string(index=False))

print("\n=== items satisfying the joint criterion in all three countries ===")
_j = cond_df[cond_df.joint].groupby("item")["country"].nunique()
print(sorted(_j[_j == len(COUNTRIES)].index.tolist()))

NLD: human=1220a39d replicate=ca9e7b8d reformulated=4d877d2e
    check JSD 1.1e-12 OK | check D 0.0e+00 OK
DEU: human=43c7f1ac replicate=362ef853 reformulated=fa0fec02
    check JSD 8.9e-12 OK | check D 0.0e+00 OK
GBR: human=a4c137d0 replicate=7f6b27ed reformulated=a96d2d22
    check JSD 4.3e-12 OK | check D 0.0e+00 OK

=== NLD: baseline JSD and signed gap D ===
                     item  jsd_base  jsd_ci_lo  jsd_ci_hi    D_base   D_ci_lo   D_ci_hi   p_D_BH fold_risk  human_collapsed  collapsed  n_human  n_silicon
            Gay Parenting  0.073546   0.061634   0.087799  0.071361  0.057685  0.084970 0.000114     False            False      False     1913       2135
           Refugee Asylum  0.084693   0.073365   0.097013  0.171020  0.152366  0.189961 0.000114     False            False      False     1546       2145
          Income Equality  0.097591   0.085537   0.113076       NaN       NaN       NaN      NaN       NaN            False      False     1977       2145
      Maternal 

### Analaysis for the most-desirable-only weighting


In [29]:
# Significance for the most-desirable-only weighting
def build_weights_mdo(entry, codes):
    """Ordinal items -> weight 1 on the single most desirable option (the scale
    endpoint), 0 elsewhere, norm 1, collapsing the neutral options into the
    undesirable side. Categorical and non-directional items are unchanged, so only
    ordinal results differ from build_weights."""
    if entry.get("kind") != "ordinal":
        return build_weights(entry, codes)
    codes = sorted(codes)
    de = entry["desirable_end"]
    return {c: (1.0 if c == de else 0.0) for c in codes}, 1.0

ORDINAL_ITEMS = [n for n, s in desirability_spec.items()
                 if s.get("kind") == "ordinal" and s.get("directional", True)]

mdo_cond = []
for _c in COUNTRIES:
    _human, _base, _ref = load_country(_c)
    _, _c2, _ = bootstrap_all(_base, {"Reformulated": {"data": _ref}}, _human,
                              name_dict, country=_c, weight_fn=build_weights_mdo)
    mdo_cond.append(_c2)
mdo_cond_df = pd.concat(mdo_cond, ignore_index=True)

def _joint_full(df):                    # Table 6 bolding: significant joint AND positive baseline gap
    return df["joint"] & (df["D_base"] > 0)

sp  = cond_df.assign(joint_full=_joint_full(cond_df)).set_index(["country", "item"])
mdo = mdo_cond_df.assign(joint_full=_joint_full(mdo_cond_df)).set_index(["country", "item"])

rows = []
for _c in COUNTRIES:
    for name in ORDINAL_ITEMS:
        if (_c, name) not in sp.index:
            continue
        s, m = sp.loc[(_c, name)], mdo.loc[(_c, name)]
        rows.append({"country": _c, "item": name,
                     "D_sp": s.D_base, "joint_sp": bool(s.joint_full),
                     "D_mdo": m.D_base, "dAbs_mdo": m.delta_absD,
                     "sig_absD_mdo": bool(m.sig_absD), "joint_mdo": bool(m.joint_full)})
comp = pd.DataFrame(rows)
print(comp.to_string(index=False,
      formatters={c: "{:+.3f}".format for c in ["D_sp", "D_mdo", "dAbs_mdo"]}))

print("\nItems whose joint-criterion label changes under most-desirable-only:")
_chg = comp[comp.joint_sp != comp.joint_mdo]
print("  none" if _chg.empty else
      _chg[["country", "item", "joint_sp", "joint_mdo", "D_sp", "D_mdo"]].to_string(index=False))

country                    item   D_sp  joint_sp  D_mdo dAbs_mdo  sig_absD_mdo  joint_mdo
    NLD           Gay Parenting +0.071     False -0.030   +0.049         False      False
    NLD     Maternal Employment +0.029     False -0.071   +0.034         False      False
    NLD      Immigration Impact +0.177     False +0.005   +0.021          True      False
    NLD           Strong Leader +0.336      True +0.597   -0.117          True       True
    NLD Government Surveillance -0.262     False -0.539   -0.049          True      False
    NLD      Immigration Policy +0.165      True -0.063   +0.000         False      False
    DEU           Gay Parenting +0.141     False +0.058   +0.010         False      False
    DEU     Maternal Employment +0.140     False +0.073   +0.196          True      False
    DEU      Immigration Impact +0.172     False +0.006   +0.065          True      False
    DEU           Strong Leader +0.245      True +0.453   -0.066          True       True
    DEU Go